# AI-Assisted Transition State Search

In this notebook, we compare classical and AI-based approaches for finding transition states (TS), with a focus on:

- Standard non-AI methods
- TS prediction (partially) using AI
- Ranking and refinement strategies with and without AI
- Practical workflows

We will:
1. Establish a classical baseline (NEB)
2. Exchange the electronic structure backend with an MLIP
3. Generate TS candidates using only AI (tsdiff)
4. Discuss refinement strategies and where AI can help

In [1]:
import torch
import torchani
import numpy as np

from ase import Atoms
from ase.mep import NEB, idpp_interpolate
from ase.optimize import BFGS
from ase.calculators.calculator import Calculator, all_changes
from xtb.ase.calculator import XTB
from ase.build import molecule

/home/marco/miniconda3/envs/tsdiff_notebook/lib/python3.13/site-packages/torchani/aev.py:16: UserWarning: cuaev not installed
  warnings.warn("cuaev not installed")


## Transition State Search: Practical Perspective

Classical methods:
- NEB: robust but expensive and initialization-sensitive

AI-based methods:
- Direct TS prediction: fast, but ranking is often imperfect

Key idea:
> The correct TS is often in the candidate set, but not ranked first.

We focus on:
- Candidate generation
- Ranking
- Efficient refinement

In [3]:
# Structure definitions for ase for a proton transfer between two water molecules
reactant = Atoms(
    symbols=["O","H","H","O","H","H"],
    positions=[
        [0.000,0.000,0.000],
        [0.757,0.586,0.000],
        [-0.757,0.586,0.000],
        [2.500,0.000,0.000],
        [3.257,0.586,0.000],
        [1.743,0.586,0.000]
    ]
)

product = Atoms(
    symbols=["O","H","H","O","H","H"],
    positions=[
        [0.000,0.000,0.000],
        [0.757,0.586,0.000],
        [1.200,0.000,0.000],
        [2.500,0.000,0.000],
        [3.257,0.586,0.000],
        [1.743,0.586,0.000]
    ]
)

In [4]:
# raw form of standard organic SN2 reaction, which might be a good example later
"""
# SN2: Cl- + CH3Br → CH3Cl + Br-

from ase.build import molecule

ch3br = molecule("CH3Br")
cl = Atoms("Cl", positions=[[3.0, 0.0, 0.0]])

reactant = ch3br + cl

# Product: Cl replaces Br
product = reactant.copy()
product.positions[-1] = product.positions[0] + np.array([-1.8, 0, 0])

# Use xTB (already uses effective core approximations internally)

attach_calc(reactant)
attach_calc(product)

# Then same NEB workflow as above
"""

'\n# SN2: Cl- + CH3Br → CH3Cl + Br-\n\nfrom ase.build import molecule\n\nch3br = molecule("CH3Br")\ncl = Atoms("Cl", positions=[[3.0, 0.0, 0.0]])\n\nreactant = ch3br + cl\n\n# Product: Cl replaces Br\nproduct = reactant.copy()\nproduct.positions[-1] = product.positions[0] + np.array([-1.8, 0, 0])\n\n# Use xTB (already uses effective core approximations internally)\n\nattach_calc(reactant)\nattach_calc(product)\n\n# Then same NEB workflow as above\n'

## Baseline: Nudged Elastic Band (NEB)

We compute a reaction path between reactant and product.

In [5]:
# It took me a long time to find something, which can use neb with an electronic
# structure backend as well as an MLIP, and setting it up was even worse!

# Anyway this runs neb with xtb
def attach_xtb(atoms):
    atoms.calc = XTB(method="GFN2-xTB")
    return atoms

def run_neb(reactant, product, method, n_images=5):
    images = [reactant.copy()]

    # this didn't work, because this kind of simple interpolation
    # yielded guesses, which were so bad that neb didn't converge
    # even with small step sizes, etc.
    #for i in range(n_images):
    #    img = reactant.copy()
    #    img.positions = reactant.positions + (i+1)/(n_images+1)*(product.positions - reactant.positions)
    #    images.append(img)

    for i in range(n_images):
        images.append(reactant.copy())

    images.append(product.copy())

    # attach calculator ("attach" methods)
    for img in images:
        method(img)

    # For some reason different calculators are recommended, but
    # for the current setup this works as well
    neb = NEB(images, allow_shared_calculator=True)

    # This does some proper interpolation for the neb images
    idpp_interpolate(images)

    # choose a rather small step size here, because it won't converge otherwise
    opt = BFGS(neb, maxstep=0.02)

    # This will literally run forever, if you don't give a max_iter
    opt.run(fmax=0.1, steps=500)

    # assume middle image to be TS for now
    return images[len(images)//2], images

ts_xtb, neb_images_xtb = run_neb(reactant, product, attach_xtb)
# TODO: output and visualize structure

/home/marco/miniconda3/envs/tsdiff_notebook/lib/python3.13/site-packages/ase/mep/neb.py:329: UserWarning: The default method has changed from 'aseneb' to 'improvedtangent'. The 'aseneb' method is an unpublished, custom implementation that is not recommended as it frequently results in very poor bands. Please explicitly set method='improvedtangent' to silence this warning, or set method='aseneb' if you strictly require the old behavior (results may vary). See: https://gitlab.com/ase/ase/-/merge_requests/3952
  warnings.warn(


      Step     Time          Energy          fmax
BFGS:    0 21:33:08     -271.190894        5.501546
BFGS:    1 21:33:08     -271.468028        5.401459
BFGS:    2 21:33:08     -271.732381        5.262142
BFGS:    3 21:33:08     -271.987200        5.047736
BFGS:    4 21:33:08     -272.232155        4.720672
BFGS:    5 21:33:08     -272.465971        4.232987
BFGS:    6 21:33:08     -272.685469        3.526028
BFGS:    7 21:33:08     -272.880933        2.759582
BFGS:    8 21:33:08     -273.022676        2.606324
BFGS:    9 21:33:09     -273.124152        2.429523
BFGS:   10 21:33:09     -273.209067        2.168336
BFGS:   11 21:33:09     -273.288113        2.064542
BFGS:   12 21:33:09     -273.365759        2.008390
BFGS:   13 21:33:09     -273.442807        1.966512
BFGS:   14 21:33:09     -273.518823        1.933089
BFGS:   15 21:33:09     -273.593343        1.904927
BFGS:   16 21:33:09     -273.666129        1.880115
BFGS:   17 21:33:09     -273.737431        1.858963
BFGS:   18 21:

In [6]:
# And the following two cells run neb with an MLIP

# setup MLIP model
# if your device has a GPU I highly recommend to use it, by setting the device here
# and previously importing the appropriate pytorch expansion into your conda environment.
# For an nvidia GPU this would be torch.device("cuda")
device = torch.device("cpu")
ani_model = torchani.models.ANI2x().to(device)
species_converter = ani_model.species_converter

# mocked calculator class for MLIP
class ANICalculator(Calculator):
    implemented_properties = ['energy', 'forces']

    def calculate(self, atoms=None, properties=['energy'], system_changes=all_changes):
        super().calculate(atoms, properties, system_changes)

        coords = torch.tensor(
            atoms.get_positions(),
            dtype=torch.float32,
            device=device,
            requires_grad=True
        ).unsqueeze(0)

        atomic_numbers = torch.tensor(
            [atoms.get_atomic_numbers()],
            dtype=torch.long,
            device=device
        )

        # This handles conversion correctly, because it uses something
        # different then atomic numbers, because machine learners have
        # no clue what they are doing!!!!!!!!!!!!!!!!!!
        species, coords = species_converter((atomic_numbers, coords))

        energy = ani_model((species, coords)).energies
        forces = -torch.autograd.grad(energy.sum(), coords)[0]

        self.results['energy'] = energy.item()
        self.results['forces'] = forces.squeeze(0).cpu().detach().numpy()


def attach_ani(atoms):
    atoms.calc = ANICalculator()
    return atoms

/home/marco/miniconda3/envs/tsdiff_notebook/lib/python3.13/site-packages/torchani/resources/


In [7]:
# run neb with MLIP
ts_ani, neb_images_ani = run_neb(reactant, product, attach_ani)
# TODO: output and visualize structure

/tmp/ipykernel_35888/3286128833.py:22: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1739474892959/work/torch/csrc/utils/tensor_new.cpp:254.)
  atomic_numbers = torch.tensor(


      Step     Time          Energy          fmax
BFGS:    0 21:33:14     -152.553632        0.267734
BFGS:    1 21:33:14     -152.556657        0.266925
BFGS:    2 21:33:14     -152.572318        0.259177
BFGS:    3 21:33:14     -152.587429        0.248928
BFGS:    4 21:33:14     -152.601326        0.234359
BFGS:    5 21:33:14     -152.613750        0.217824
BFGS:    6 21:33:15     -152.624507        0.200151
BFGS:    7 21:33:15     -152.633145        0.170635
BFGS:    8 21:33:15     -152.639143        0.144718
BFGS:    9 21:33:15     -152.642468        0.159477
BFGS:   10 21:33:15     -152.642578        0.187401
BFGS:   11 21:33:15     -152.640523        0.196690
BFGS:   12 21:33:15     -152.634305        0.279260
BFGS:   13 21:33:15     -152.627976        0.354058
BFGS:   14 21:33:15     -152.624607        0.389794
BFGS:   15 21:33:16     -152.624489        0.389545
BFGS:   16 21:33:16     -152.626003        0.367257
BFGS:   17 21:33:16     -152.627478        0.339821
BFGS:   18 21:

In [ ]:
# Here I found an other tool, which seems to automatically do a setup
# for conformers and runs them all in one go. Note that this does NOT
# make use of any AI tools, but it might be useful for something we might
# want to do later.

import autode as ade
from autode.species import Reactant, Product
from autode.reactions import Reaction

r = Reactant(name="r", atoms=[ade.Atom(a.symbol, *a.position) for a in reactant])
p = Product(name="p", atoms=[ade.Atom(a.symbol, *a.position) for a in product])

rxn = Reaction(r, p)
ts_autode = rxn.locate_transition_state()
ts_autode.print_xyz_file("ts_autode.xyz")

# this is so stupid, don't even get me started on this...
ade.Config.n_cores = 8
ade.Config.max_core = 3000

# Create an 8 image nudged elastic band with intermediate images interpolated
# from the final points, thus they must be structurally similar
neb = ade.CINEB.from_end_points(reac, prod, num=8)
# minimise with XTB
neb.calculate(method=ade.methods.XTB(), n_cores=4)

print("Found a peak: ", neb.images.contains_peak)

In [ ]:
# print the geometry of the peak species
neb.peak_species.print_xyz_file(filename="peak.xyz")

#print(dir(ade.methods))

neb.peak_species.atoms

# Direct TS Prediction

Instead of searching along a path, we generate TS candidates directly using an AI model.

## How does it work?

tsdiff uses a diffusion model (hence the name). Diffusion models are very well suited for this task, because multiple transition states are often close to each other in energy even though they sometimes belong to different reaction classes, e.g. substitution vs. elimination. The diffusion model will start from a random guess, which it prints out and is very far away from any sensible chemical structure, and in each iteration improves the structures through denoising it. Think about it as the network actively diffusing the input towards the output (again hence the name). On top of that tsdiff does a good job of identifying reaction-patterns, by mapping the molecule onto a 2D graph.

Now we are testing it for a proton transfer between hydronium and water as well as a proton transfer between CH4 and CH3.

If you have a GPU I once again highly recommend using it, by setting the pytorch device.

In [1]:
# create train, validation and test set, but in this case
# we are only interested in testing custom cases, so only
# the test set will contain data.

from helpers.preprocessing_helper import preprocess_args, preprocess
from helpers.sampling_helper import sampling_args, sample

In [2]:
# class attributes contain parameters
prep_params = preprocess_args()
prep_params.train = 0.0
prep_params.valid = 0.0
# generate preprocessed files
preprocess(prep_params)

2it [00:00, 402.56it/s]


In [3]:
# run a pretrained model on the generated test data
sample_params = sampling_args()
#sample_params.device = "cuda"
# only use one model for faster calculation
# if you want to use all 8 available ones, just uncomment this line
sample_params.ckpt = [sample_params.ckpt[0]]
# default in tsdiff is 5000
sample_params.n_steps = 1000
sample(sample_params)

[2026-05-06 16:13:20,923::test::INFO] <helpers.sampling_helper.sampling_args object at 0x76b75b15c610>
[2026-05-06 16:13:20,924::test::INFO] Loading model...
[2026-05-06 16:13:20,965::test::INFO] load model from logs/trained_ckpt/ens0/checkpoints/best_ckpt.pt
[2026-05-06 16:13:21,015::test::INFO] Loading datasets...
[2026-05-06 16:13:21,016::test::INFO] Test file from data/scratch/random_split_42/test_data.pkl.
 Loading dataset...


0it [00:00, ?it/s]

Initial Position
tensor([[-1.1910e+01, -1.8440e+01, -1.0005e+01],
        [ 1.4768e+00,  8.7110e-01, -1.0608e+00],
        [-3.0840e-01, -1.9795e+01,  2.7137e+00],
        [-2.0377e+01, -1.7918e+01,  1.1829e+01],
        [-2.6667e+00, -1.1435e+01, -2.1607e+01],
        [-7.6162e+00, -1.3511e+01,  1.4469e+01],
        [ 1.6707e+01,  5.9802e+00,  4.3551e+00],
        [-2.0503e+00, -1.0440e+01, -1.2867e+01],
        [ 1.0295e+01,  1.8406e+01,  3.5174e-01],
        [-1.9278e+00,  2.2354e+01,  8.1388e+00],
        [-2.1630e+01,  2.4000e-02, -2.6832e+01],
        [ 8.3482e+00,  1.1577e+01, -3.0906e+00],
        [-8.9028e+00,  4.3331e+00, -1.2419e+01],
        [-1.7825e+01, -2.8411e+00, -5.3819e+00],
        [-1.0890e+01, -8.5693e+00,  2.2328e+00],
        [ 4.4067e+00, -1.3771e+01,  1.8083e+00]])


sample: 0it [00:00, ?it/s]

Generated Position
tensor([[-0.3192, -0.1920, -1.2261],
        [ 0.0808,  0.2305,  1.2386],
        [ 0.3298,  0.2233, -1.8440],
        [-0.6138, -1.1082, -1.4231],
        [-0.3337,  1.0021,  1.6612],
        [ 0.9171, -0.1429,  1.6195],
        [-0.0609, -0.0127, -0.0261],
        [ 0.7046,  0.7525,  0.9829],
        [ 1.3152,  0.1411,  1.5957],
        [-0.0659,  1.3425,  1.4319],
        [ 1.3323,  1.3842,  0.2665],
        [-0.0075, -0.0566,  0.0341],
        [-0.7088, -0.7680, -0.9431],
        [-1.5964, -1.1714, -0.4363],
        [-0.9449, -0.0613, -1.7116],
        [-0.0286, -1.5630, -1.2202]])
tensor([[-0.3192, -0.1920, -1.2261],
        [ 0.0808,  0.2305,  1.2386],
        [ 0.3298,  0.2233, -1.8440],
        [-0.6138, -1.1082, -1.4231],
        [-0.3337,  1.0021,  1.6612],
        [ 0.9171, -0.1429,  1.6195],
        [-0.0609, -0.0127, -0.0261],
        [ 0.7046,  0.7525,  0.9829],
        [ 1.3152,  0.1411,  1.5957],
        [-0.0659,  1.3425,  1.4319],
        [ 1.3323, 

In [1]:
# visualize the generated transition state structures
import py3Dmol

def view_xyz(filename):
    with open(filename, "r") as f:
        xyz = f.read()

        viewer = py3Dmol.view(width=500, height=400)
        viewer.addModel(xyz, "xyz")

        # nice default style
        viewer.setStyle({
            "stick": {"radius": 0.15},
            "sphere": {"scale": 0.3}
        })

        viewer.zoomTo()
    return viewer

view_xyz("ts0.xyz")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [2]:
view_xyz("ts1.xyz")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Clustering and Training

A nice feature of the diffusion methodology is that when running this multiple times, different transition state structures will be found, if the PES is reasonably complex. If that is the case you can use tsdiffs clustering tool, which groups the reactions e.g. into competing reaction channels. Our test reactions are, however, too simple for that.

Regarding training there is, to the best of my knowledge, no short retraining opportunity for tsdiff, so if you want to add something you have to fully retrain from your seed.

If you are interested you can find the GitHub repository for tsdiff [here](https://github.com/seonghann/tsdiff).

### Refinement of the guessed structures

Standard: Use TS guess and optimize purely with electronic structure backend

In [ ]:
def refine_xtb(ts):
    ts = ts.copy()
    attach_xtb(ts)
    opt = BFGS(ts)
    opt.run(fmax=0.05)
    return ts

#ts_refined_std = refine_xtb(ranked[0][0])

Surrogate accelaration by preoptimizing with MLIP

In [ ]:
def refine_ml_then_xtb(ts):
    ts = ts.copy()
    
    # Step 1: ANI refinement
    attach_ani(ts)
    opt = BFGS(ts)
    opt.run(fmax=0.1)
    
    # Step 2: xTB refinement
    attach_xtb(ts)
    opt = BFGS(ts)
    opt.run(fmax=0.05)
    
    return ts

#ts_refined_ml = refine_ml_then_xtb(ranked[0][0])

Accelaration of initial Hessian guess (using MLIP)

In [ ]:
# For this I found that Hessians are typically build from MLIP information

## Ranking TS Candidates

We rank candidates based on approximate energies or heuristics.

Observation:
- Top-1 is often wrong
- Correct TS may appear in top-2 or top-3

## Uncertainty via Conformer Ensembles

# Allegedly this is what autode does, but I think we might just throw this boy out along with the autode stuff

We generate multiple conformers and predict TS for each.

Observation:
- Low spread → reliable
- High spread → unreliable

## Where Do TS Predictors Work Well?

| System Type | Performance |
|------------|------------|
| Small organic reactions | Excellent |
| Proton transfer | Very strong |
| SN2 reactions | Strong |
| Rearrangements | Mixed |
| Large flexible systems | Challenging |
| Transition metals | Poor |
| Charged/solvent systems | Difficult |

Conclusion:
AI TS predictors are powerful but require hybrid workflows and validation.

# Now try tsdiff for your own reaction

Now for the fun part, where you create your own reaction and can use tsdiff to predict the transition state structure.

Therefore you will have to do the following:
- Create SMARTS for the reactions you are interested in, e.g. with ChatGPT
- Create xyz structure of reactants/products/whatever (this is just for workflow consistency with tsdiff)
- Create your new "test set" with the preprocessing routine
- Generate the transition state structure using the sampling routine

You can just append smarts.csv and structures.xyz in data/raw_data/ with the new smarts and xyz, but **beware of the atom ordering** in the smarts and the xyz! The atoms in the smarts need to have the same enumeration for reactants and products, and this ordering also needs to apply for the xyz coordinates. To give an example, if the reactant side of the smart has an O at position 2 and H at position three, but it's vice versa on the product side, this won't work.